In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [2]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')
PROJECT_DIR = DATA_DIR + "/"

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

SAVE_DIR = DATA_DIR + "/datasets/"

seg = pd.read_csv(SAVE_DIR + "segment_features_manchester.csv")

print(f"Loaded {len(seg):,} segments")
print(f"Segments with 0 accidents: this is ALL segments with accidents")
print(f"Accident distribution:\n{seg['total_accidents'].describe()}")

Loaded 10,895 segments
Segments with 0 accidents: this is ALL segments with accidents
Accident distribution:
count    10895.000000
mean         1.555668
std          8.790151
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        688.000000
Name: total_accidents, dtype: float64


In [4]:
# 1. Flaging low-sample segments (fewer than 3 accidents --> unreliable rate)
seg['low_sample'] = (seg['total_accidents'] < 3).astype(int)
print(f"\nLow sample segments (<3 accidents): {seg['low_sample'].sum():,} ({seg['low_sample'].mean()*100:.1f}%)")


Low sample segments (<3 accidents): 10,317 (94.7%)


In [5]:
# 2. Bayesian-smoothed serious rate
# Raw serious_rate is unreliable for low-accident segments
# Smoothing toward the global mean using a prior of 10 "pseudo-observations"
global_serious_rate = seg['serious_accidents'].sum() / seg['total_accidents'].sum()
prior_strength = 10  # pseudo-observations

seg['serious_rate_smoothed'] = (
    (seg['serious_accidents'] + prior_strength * global_serious_rate) /
    (seg['total_accidents'] + prior_strength)
)

print(f"\nGlobal serious rate: {global_serious_rate:.3f}")
print(f"Smoothed rate range: {seg['serious_rate_smoothed'].min():.3f} → {seg['serious_rate_smoothed'].max():.3f}")


Global serious rate: 0.260
Smoothed rate range: 0.118 → 0.440


In [6]:
# 3. Building composite risk score
# Combine: accident frequency + serious rate + road characteristics
scaler = MinMaxScaler()

# Normalise each component to 0-1
seg['freq_score']    = scaler.fit_transform(seg[['accidents_per_year']])
seg['severity_score'] = scaler.fit_transform(seg[['serious_rate_smoothed']])
seg['speed_score']   = scaler.fit_transform(seg[['avg_speed_limit']])

# Weighted composite (we'll tune these weights with the ML model later)
seg['risk_score'] = (
    0.50 * seg['freq_score'] +       # frequency matters most
    0.35 * seg['severity_score'] +   # severity second
    0.15 * seg['speed_score']        # speed as modifier
)

# Scale final score to 0-100 for readability
seg['risk_score_100'] = (seg['risk_score'] * 100).round(1)

In [7]:
# 4. Defining Risk tiers for driver alerts
def risk_tier(score):
    if score >= 70: return 'CRITICAL'
    if score >= 40: return 'HIGH'
    if score >= 15: return 'MEDIUM'
    return 'LOW'

seg['risk_tier'] = seg['risk_score_100'].apply(risk_tier)

print(f"\n Risk tier distribution ")
print(seg['risk_tier'].value_counts())

print(f"\n Top 15 highest risk segments ")
print(seg.nlargest(15, 'risk_score_100')[[
    'edge_id', 'total_accidents', 'accidents_per_year',
    'serious_rate_smoothed', 'avg_speed_limit',
    'risk_score_100', 'risk_tier'
]].to_string())

print(f"\n Risk score distribution ")
print(seg['risk_score_100'].describe())


 Risk tier distribution 
risk_tier
MEDIUM    9434
LOW       1437
HIGH        24
Name: count, dtype: int64

 Top 15 highest risk segments 
       edge_id  total_accidents  accidents_per_year  serious_rate_smoothed  avg_speed_limit  risk_score_100 risk_tier
4680     70103              688               137.6               0.178509        35.116279            61.1      HIGH
144        197                3                 0.6               0.430736        70.000000            49.1      HIGH
525       1954                5                 1.0               0.439971        62.000000            47.9      HIGH
124        164                6                 1.2               0.412473        70.000000            47.4      HIGH
130        170                6                 1.2               0.412473        70.000000            47.4      HIGH
564       2094                6                 1.2               0.412473        70.000000            47.4      HIGH
1602     16821               65    

Fom the risk score distribution results above, we can note that Low-sample, high-speed segments are dominating the HIGH tier. Look at edge 197 — only 3 accidents total, but it scores HIGH because it's a 70mph road. Meanwhile our genuinely dangerous hotspot (edge 70102, 688 accidents) only scores 61.1 — barely HIGH. The frequency component is being drowned out by speed and severity on sparse segments.
Also: No CRITICAL segments at all, and 87% are MEDIUM. Our scoring weights need rebalancing,  frequency should dominate much more strongly.


In [8]:
global_serious_rate = 0.233  # from our EDA
prior_strength = 10

seg['serious_rate_smoothed'] = (
    (seg['serious_accidents'] + prior_strength * global_serious_rate) /
    (seg['total_accidents'] + prior_strength)
)


In [9]:
# Rebalanced scoring
# Problem: raw accidents_per_year is skewed by one mega-hotspot (137/yr)
# Solution: use log scale for frequency so it doesn't get crushed

seg['log_freq'] = np.log1p(seg['accidents_per_year'])

scaler = MinMaxScaler()
seg['freq_score']     = scaler.fit_transform(seg[['log_freq']])
seg['severity_score'] = scaler.fit_transform(seg[['serious_rate_smoothed']])
seg['speed_score']    = scaler.fit_transform(seg[['avg_speed_limit']])

# New weights: frequency dominates, severity second, speed is minor modifier
seg['risk_score'] = (
    0.60 * seg['freq_score'] +
    0.30 * seg['severity_score'] +
    0.10 * seg['speed_score']
)

seg['risk_score_100'] = (seg['risk_score'] * 100).round(1)

In [10]:
# Percentile-based tiers (more robust than fixed thresholds)
p75 = seg['risk_score_100'].quantile(0.75)
p90 = seg['risk_score_100'].quantile(0.90)
p97 = seg['risk_score_100'].quantile(0.97)

print(f"Tier thresholds → MEDIUM: >{p75:.1f}, HIGH: >{p90:.1f}, CRITICAL: >{p97:.1f}")

def risk_tier(score):
    if score > p97: return 'CRITICAL'
    if score > p90: return 'HIGH'
    if score > p75: return 'MEDIUM'
    return 'LOW'

seg['risk_tier'] = seg['risk_score_100'].apply(risk_tier)

print(f"\nRisk tier distribution ")
print(seg['risk_tier'].value_counts())
print(f"\nTop 15 highest risk segments ")
print(seg.nlargest(15, 'risk_score_100')[[
    'edge_id', 'total_accidents', 'accidents_per_year',
    'serious_rate_smoothed', 'avg_speed_limit',
    'risk_score_100', 'risk_tier'
]].to_string())

print(f"\n Risk score distribution")
print(seg['risk_score_100'].describe())

Tier thresholds → MEDIUM: >19.9, HIGH: >20.4, CRITICAL: >26.4

Risk tier distribution 
risk_tier
LOW         8287
MEDIUM      1988
CRITICAL     324
HIGH         296
Name: count, dtype: int64

Top 15 highest risk segments 
       edge_id  total_accidents  accidents_per_year  serious_rate_smoothed  avg_speed_limit  risk_score_100 risk_tier
4680     70103              688               137.6               0.178123        35.116279            69.3  CRITICAL
4337     64713              272                54.4               0.277766        35.588235            67.5  CRITICAL
1602     16821               65                13.0               0.364400        65.076923            64.5  CRITICAL
7771    143899              301                60.2               0.197203        37.674419            61.4  CRITICAL
2977     40907               57                11.4               0.333284        53.333333            57.6  CRITICAL
3679     55719              168                33.6               0.21

In [12]:
# Adding a confidence flag
# Segments with few accidents have uncertain scores — flag for the model
seg['confidence'] = pd.cut(seg['total_accidents'],
                            bins=[0, 3, 10, 50, np.inf],
                            labels=['very_low', 'low', 'medium', 'high'])

print(f"\n Confidence distribution ")
print(seg['confidence'].value_counts())


 Confidence distribution 
confidence
very_low    10651
low           197
medium         35
high           12
Name: count, dtype: int64


In [13]:
#Save
seg.to_csv(SAVE_DIR + "segment_risk_scores.csv", index=False)